In [1]:
import numpy as np
import pandas as pd
from scipy.stats import zscore


In [2]:

base_path = "./Dataset_and_Description/Oxy-files/"
fs = 10.1725   # sampling rate

male_subject_ids = [2, 9, 10, 11, 13, 14, 15, 18, 19, 20]

In [3]:

def create_state_labels(n_samples, fs):
    labels = np.zeros(n_samples, dtype=int)  # 0 = Rest
    
    idx = int(30 * fs)   # start after initial 30 sec rest
    
    for _ in range(10):
        grip_start = idx
        grip_end = idx + int(10 * fs)
        
        grip_start = min(grip_start, n_samples)
        grip_end   = min(grip_end, n_samples)
        
        labels[grip_start:grip_end] = 1  # Grip = 1
        
        idx = grip_end + int(20 * fs)
        if idx >= n_samples:
            break
    
    return labels

In [4]:
from scipy.signal import butter, filtfilt
def bandpass_filter(data, fs, low=0.01, high=0.2, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

In [5]:
final_dfs = []
only_features=[]

for subject_id in range(1, 16):   # oxy1 → oxy20
    
    # ---------- READ FILE ----------
    file_path = f"{base_path}oxy{subject_id}.csv"
    df = pd.read_csv(file_path)
    
    # ---------- CLEAN ----------
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # df_filtered = df.copy()
    for col in df.columns:
        df[col] = bandpass_filter(df[col].values, fs)

    df = df.rolling(window=5, center=True).median()
    df = df.dropna()
    
    # ---------- SELECT CHANNELS SAFELY ----------
    # channel_cols = [col for col in df.columns if col.startswith("ch")]
    # channel_data = df[channel_cols]
    
    # ---------- ✅ Z-SCORE NORMALIZATION (PER CHANNEL, PER SUBJECT) ----------
    df = df.apply(zscore)
    
    # ---------- ✅ CREATE STATE ----------
    df["State"] = create_state_labels(len(df), fs)
    
    # ---------- ✅ ROW-WISE FEATURES (ON NORMALIZED DATA) ----------
    df["Mean_Row"]   = df.mean(axis=1)
    df["Std_Row"]    = df.std(axis=1)
    df["Energy_Row"] = (df** 2).sum(axis=1)
    df["RMS_Row"]    = np.sqrt((df ** 2).mean(axis=1))
    
    # ---------- ✅ ADD GENDER ----------
    if subject_id in male_subject_ids:
        df["Gender"] = 1   # Male
    else:
        df["Gender"] = 0   # Female
    
    # ---------- ✅ KEEP ONLY FINAL ML COLUMNS ----------
    final_df = df[[
        "Mean_Row",
        "Std_Row",
        "Energy_Row",
        "RMS_Row",
        "State",
        "Gender"
    ]]
    
    final_dfs.append(df)
    only_features.append(final_df)

In [6]:
final_dfs_test = []
only_features_test=[]

for subject_id in range(16, 21):   # oxy1 → oxy20
    
    # ---------- READ FILE ----------
    file_path = f"{base_path}oxy{subject_id}.csv"
    df = pd.read_csv(file_path)
    
    # ---------- CLEAN ----------
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # df_filtered = df.copy()
    for col in df.columns:
        df[col] = bandpass_filter(df[col].values, fs)

    df = df.rolling(window=5, center=True).median()
    df = df.dropna()
    
    # ---------- SELECT CHANNELS SAFELY ----------
    # channel_cols = [col for col in df.columns if col.startswith("ch")]
    # channel_data = df[channel_cols]
    
    # ---------- ✅ Z-SCORE NORMALIZATION (PER CHANNEL, PER SUBJECT) ----------
    df = df.apply(zscore)
    
    # ---------- ✅ CREATE STATE ----------
    df["State"] = create_state_labels(len(df), fs)
    
    # ---------- ✅ ROW-WISE FEATURES (ON NORMALIZED DATA) ----------
    df["Mean_Row"]   = df.mean(axis=1)
    df["Std_Row"]    = df.std(axis=1)
    df["Energy_Row"] = (df** 2).sum(axis=1)
    df["RMS_Row"]    = np.sqrt((df ** 2).mean(axis=1))
    
    # ---------- ✅ ADD GENDER ----------
    if subject_id in male_subject_ids:
        df["Gender"] = 1   # Male
    else:
        df["Gender"] = 0   # Female
    
    # ---------- ✅ KEEP ONLY FINAL ML COLUMNS ----------
    final_df = df[[
        "Mean_Row",
        "Std_Row",
        "Energy_Row",
        "RMS_Row",
        "State",
        "Gender"
    ]]
    
    final_dfs_test.append(df)
    only_features_test.append(final_df)

In [7]:
final_dataset = pd.concat(final_dfs, ignore_index=True)
final_dataset_test = pd.concat(final_dfs_test, ignore_index=True)


print("✅ FINAL DATASET SHAPE:", final_dataset.shape)

✅ FINAL DATASET SHAPE: (51660, 26)


In [8]:
final_dataset.head()

,ch1,ch2,ch3,ch4,ch5,ch6,ch7,ch8,ch9,ch10,...,ch17,ch18,ch19,ch20,State,Mean_Row,Std_Row,Energy_Row,RMS_Row,Gender
0,0.066503,-0.100691,0.049500,0.675047,0.243700,0.911247,0.202707,-0.165607,0.173002,0.505412,...,0.290500,0.640507,0.263046,0.048925,0,0.240190,0.274402,2.925736,0.691787,0
1,0.085629,-0.072208,0.079822,0.718774,0.263597,0.987105,0.250586,-0.172465,0.208833,0.564219,...,0.338784,0.685072,0.305551,0.063335,0,0.272498,0.291644,3.504852,0.811091,0
2,0.104044,-0.044105,0.110033,0.762034,0.283589,1.062907,0.298415,-0.179036,0.244501,0.622519,...,0.386852,0.729381,0.347959,0.077538,0,0.304774,0.309400,4.149548,0.943581,0
3,0.121609,-0.016520,0.140061,0.804719,0.303732,1.138529,0.346135,-0.185191,0.279916,0.680073,...,0.434553,0.773274,0.390138,0.091438,0,0.336975,0.327470,4.857352,1.088792,0
4,0.138192,0.010422,0.169834,0.846720,0.324082,1.213846,0.393685,-0.190804,0.314996,0.736651,...,0.481745,0.816599,0.431955,0.104941,0,0.369055,0.345673,5.625219,1.246133,0


In [9]:
# import matplotlib.pyplot as plt
# import numpy as np

# rows_per_subject = 3448

# # ✅ Only male rows
# male_df = final_dataset[final_dataset["Gender"] == 0].reset_index(drop=True)

# for i in range(10):   # 10 male subjects
    
#     start = i * rows_per_subject
#     end   = (i + 1) * rows_per_subject
    
#     subj_df = male_df.iloc[start:end]
    
#     rest_vals = subj_df[subj_df["State"] == 0]["Mean_Row"]
#     grip_vals = subj_df[subj_df["State"] == 1]["Mean_Row"]
    
#     plt.figure()
#     plt.boxplot([rest_vals, grip_vals])
#     plt.xticks([1, 2], ["Rest", "Grip"])
#     plt.title(f"Male Subject {i+1} – Mean_Row")
#     plt.ylabel("Mean_Row")
#     plt.grid(True)
#     plt.show()

In [10]:
X = final_dataset.drop(columns=["State"])
y = final_dataset["State"]
gender = final_dataset["Gender"]

In [11]:
X_test = final_dataset_test.drop(columns=["State"])
y_test = final_dataset_test["State"]
gender_test = final_dataset_test["Gender"]

In [12]:
from sklearn.model_selection import train_test_split

# ✅ 1) First split: 60% Train | 40% Temp  (stratified by Gender)
X_train, X_val, y_train, y_val, gender_train, gender_val = train_test_split(
    X, y, gender,
    test_size=0.2,
    random_state=42,
    stratify=gender
)

# # ✅ 2) Second split: 20% Val | 20% Test (also stratified by Gender)
# X_val, X_test, y_val, y_test, gender_val, gender_test = train_test_split(
#     X_temp, y_temp, gender_temp,
#     test_size=0.5,   # half of 40% → 20% & 20%
#     random_state=42,
#     stratify=gender_temp
# )


In [13]:
print("TRAIN Gender Ratio:\n", gender_train.value_counts(normalize=True))
print("\nVAL Gender Ratio:\n", gender_val.value_counts(normalize=True))
print("\nTEST Gender Ratio:\n", gender_test.value_counts(normalize=True))

TRAIN Gender Ratio:
 Gender
0    0.533343
1    0.466657
Name: proportion, dtype: float64

VAL Gender Ratio:
 Gender
0    0.533295
1    0.466705
Name: proportion, dtype: float64

TEST Gender Ratio:
 Gender
1    0.6
0    0.4
Name: proportion, dtype: float64


In [14]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled   = scaler.transform(X_val)
# X_test_scaled  = scaler.transform(X_test)

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {
    "C": [0.1, 1, 10],
    "gamma": [0.01, 0.1, 1, "scale"],
    "kernel": ["rbf"]
}

svm = SVC(class_weight="balanced")

grid = GridSearchCV(
    svm,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=3,      # ✅ THIS SHOWS TRAINING LIVE
    return_train_score=True
)

# grid.fit(X_train, y_train)

In [26]:
# from sklearn.metrics import accuracy_score

# y_val_pred = grid.predict(X_val)
# val_acc = accuracy_score(y_val, y_val_pred)

# print("✅ Validation Accuracy:", val_acc)


In [25]:
# from sklearn.metrics import classification_report, confusion_matrix

# y_test_pred = grid.predict(X_test)

# print("✅ Test Accuracy:", accuracy_score(y_test, y_test_pred))
# print("\n✅ Classification Report:\n", classification_report(y_test, y_test_pred))
# print("\n✅ Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

In [ ]:
# best_params = grid.best_params_
# print("✅ Best Parameters Found:", best_params)

best_svm = SVC(
    kernel="rbf",
    C=1,
    gamma=0.1,
    class_weight="balanced"
)

best_svm.fit(X_train, y_train)


,C,5
,kernel,'rbf'
,degree,3
,gamma,0.5
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [23]:
from sklearn.metrics import accuracy_score

y_val_pred = best_svm.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)

print("✅ Validation Accuracy:", val_acc)


✅ Validation Accuracy: 0.9691250483933411


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred = best_svm.predict(X_test)

print("✅ Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\n✅ Classification Report:\n", classification_report(y_test, y_test_pred))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

✅ Test Accuracy: 0.8488385598141696

✅ Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.95      0.90     12170
           1       0.83      0.61      0.70      5050

    accuracy                           0.85     17220
   macro avg       0.84      0.78      0.80     17220
weighted avg       0.85      0.85      0.84     17220


✅ Confusion Matrix:
 [[11516   654]
 [ 1949  3101]]


In [21]:
import pickle

# Save SVM model
with open("svm_model_5_test.pkl", "wb") as f:
    pickle.dump(best_svm, f)

print("✅ SVM model saved as svm_model.pkl")


✅ SVM model saved as svm_model.pkl


In [28]:
with open("svm_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ Scaler saved as svm_scaler.pkl")


✅ Scaler saved as svm_scaler.pkl


In [29]:
print("Train Accuracy:", best_svm.score(X_train_scaled, y_train))
print("Val Accuracy:", best_svm.score(X_val_scaled, y_val))

Train Accuracy: 0.9257888114595432
Val Accuracy: 0.9170296167247387
